## Unsloth

In [ ]:

# ============================================================================
# §0. Global imports & constants
# ============================================================================
import os, sys, glob, subprocess, site, shutil, stat, json, zipfile
import importlib.util

os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

# ============================================================================
# §1. Install Triton wheel from /kaggle/input into a private target dir.
# ============================================================================
candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)
if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")

_TRITON_TARGET = "/kaggle/working/pydeps"
os.makedirs(_TRITON_TARGET, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "--no-deps", "--target", _TRITON_TARGET,
     "--upgrade", "--ignore-installed", candidates[0]],
    check=True,
)
if _TRITON_TARGET not in sys.path:
    sys.path.insert(0, _TRITON_TARGET)
site.addsitedir(_TRITON_TARGET)
print(f"Custom target added: {_TRITON_TARGET}")
print("triton spec:", importlib.util.find_spec("triton"))

# ============================================================================
# §2. Blackwell (B200) ptxas patch — point triton at the sm_100-aware ptxas
#     shipped in ryanholbrook/nvidia_utility_script and pin the reported
#     CUDA version to 12.0 so triton's version check passes.
# ============================================================================
sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')
ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
ptxas_dst = '/tmp/ptxas-blackwell'
if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
    shutil.copy2(ptxas_src, ptxas_dst)
    os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    src_bin = os.path.dirname(ptxas_src)
    dst_bin = '/tmp/triton_nvidia_bin'
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst
    # Fake the package __file__ so triton resolves ptxas in our dst_bin.
    import triton.backends.nvidia as nv_backend
    nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
    os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

import triton.backends.nvidia.compiler as nv_compiler
nv_compiler.get_ptxas_version = lambda arch: '12.0'
print('Training environment fixes applied.')

# ============================================================================
# §3. Offline install of unsloth + trl + mamba_ssm + causal-conv1d wheels
# ============================================================================
import torch
print(f"Python: {sys.version.split()[0]}  Torch: {torch.__version__}  "
      f"CUDA: {torch.version.cuda}  CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("Training requires a GPU runtime (Nemotron uses CUDA wheels).")

packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
if not os.path.isdir(packages_dir):
    raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--no-index", "--find-links", packages_dir,
     "unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes"],
    check=True,
)

# mamba_ssm + causal-conv1d ship as separate wheels under /kaggle/input.
all_mamba  = sorted(glob.glob("/kaggle/input/**/mamba_ssm-*.whl",    recursive=True))
all_causal = sorted(glob.glob("/kaggle/input/**/causal*conv1d*.whl", recursive=True))
causal_wheel = all_causal[-1] if all_causal else None
mamba_wheel  = all_mamba[-1]  if all_mamba  else None
print(f"Selected causal wheel: {causal_wheel}")
print(f"Selected mamba wheel:  {mamba_wheel}")
if causal_wheel:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
if mamba_wheel:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
else:
    raise FileNotFoundError("Could not find a compatible mamba_ssm wheel under /kaggle/input.")
print("Offline package installation finished.")

# ============================================================================
# §4. Load Nemotron-3-Nano base model and wrap with a trainable LoRA
# ============================================================================
import kagglehub
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 8192
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
print(f"Model path: {MODEL_PATH}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=False,
    trust_remote_code=True,
    unsloth_force_compile=False,
    attn_implementation="sdpa",
    dtype=torch.bfloat16,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

LORA_RANK    = 32
LORA_ALPHA   = 32
LORA_DROPOUT = 0.0
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "in_proj", "out_proj", "up_proj", "down_proj",
]
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=target_modules,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

# --- Sanity check: confirm which modules our target_modules pattern actually
#     hooked. For MoE models (Nemotron-3-Nano is 30B-A3B with experts), expert
#     submodules may live under names like `experts.<i>.{up,down,gate}_proj`,
#     which would still match the substrings above — print to verify.
print("\n=== Modules matching target_modules patterns ===")
matched = []
for name, _ in model.named_modules():
    if any(t in name for t in target_modules):
        matched.append(name)
        print(name)
print(f"\nTotal matched module names: {len(matched)}")

# Also count how many actually got a LoRA adapter attached (peft renames them
# to `<name>.lora_A` / `<name>.lora_B`):
lora_layers = [n for n, _ in model.named_modules() if n.endswith(".lora_A.default")]
print(f"LoRA-adapted layers: {len(lora_layers)}")
# Show coverage by family (q_proj, up_proj, etc.)
from collections import Counter
fam_counts = Counter()
for n in lora_layers:
    base = n.replace(".lora_A.default", "")
    leaf = base.rsplit(".", 1)[-1]
    fam_counts[leaf] += 1
print("LoRA layer counts by leaf module name:", dict(fam_counts))

# ============================================================================
# §5. Build dataset, configure SFT, run training, save adapter
# ============================================================================
import re, math, time, random
import pandas as pd
import torch.nn.functional as F
from collections import defaultdict
from torch.utils.data import DataLoader, Sampler
from datasets import Dataset as HFDataset
from trl import SFTTrainer, SFTConfig

SEED = 42
PROMPT_LOSS_WEIGHT = 0.1  # Set to 0.0 to ignore prompt loss, 1.0 for normal full-text SFT loss.
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
ASSISTANT_PREFIX = "<|im_start|>assistant\n"

DATASET_PATH = "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv"
df = pd.read_csv(DATASET_PATH)
train_df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"Full dataset: {len(train_df)} rows")

# --- 5.1 Build SFT records: strip any \boxed{...} and any think tags
#         from the source CoT, then append only the gold boxed answer.
records = []
record_types = []
for _, row in train_df.iterrows():
    prompt = str(row["prompt"])
    answer = str(row["answer"])
    cot    = str(row["generated_cot"])
    if not cot or cot == "nan" or len(cot.strip()) < 5:
        continue
    cot_cleaned = re.sub(r'\\boxed\{[^}]*\}', '', cot)
    cot_cleaned = re.sub(r'\s*</?think>\s*', '\n', cot_cleaned).strip()
    user_content      = prompt + PROMPT_SUFFIX
    assistant_content = (cot_cleaned + "\n" if cot_cleaned else "") + f"\\boxed{{{answer}}}"
    records.append({"messages": [
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]})
    record_types.append(str(row["type"]))
dataset = HFDataset.from_list(records)
print(f"SFT records: {len(records)}")

# --- 5.2 Pre-apply the chat template without adding think tags.
def formatting_prompts_func(example):
    texts = []
    for conversation in example["messages"]:
        try:
            text = tokenizer.apply_chat_template(
                conversation, tokenize=False,
                add_generation_prompt=False, enable_thinking=False,
            )
        except TypeError:
            text = tokenizer.apply_chat_template(
                conversation, tokenize=False, add_generation_prompt=False,
            )
        texts.append(text)
    return {"text": texts}

print("Materializing text column before training...")
dataset = dataset.map(
    formatting_prompts_func, batched=True, num_proc=4,
    desc="Applying chat template",
)

# --- 5.3 Training config. NOTE: max_grad_norm=1e9 effectively disables gradient
#         clipping to mirror the nemotron-master reference recipe.
training_args = SFTConfig(
    output_dir="/kaggle/working/sft_output",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_length=8192,
    adam_beta1=0.9, adam_beta2=0.95, adam_epsilon=1e-8,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=2,
    remove_unused_columns=False,
    seed=SEED,
    report_to="none",
    packing=False,
)

# --- 5.4 Stratified-by-type batching (approximates nemotron-master).
def build_stratified_index_order(labels, batch_size, seed):
    by_label = defaultdict(list)
    for idx, label in enumerate(labels):
        by_label[label].append(idx)
    rng = random.Random(seed)
    for idx_list in by_label.values():
        rng.shuffle(idx_list)
    n_batches = max(1, math.ceil(len(labels) / batch_size))
    batches = [[] for _ in range(n_batches)]
    batch_order = list(range(n_batches))
    rng.shuffle(batch_order)
    assigned = 0
    for label in sorted(by_label.keys()):
        for idx in by_label[label]:
            batches[batch_order[assigned % n_batches]].append(idx)
            assigned += 1
    order = [idx for batch in batches for idx in batch]
    if len(order) != len(labels):
        raise ValueError("Stratified order size mismatch")
    return order

class PrecomputedOrderSampler(Sampler):
    def __init__(self, order):
        self.order = list(order)
    def __iter__(self):
        return iter(self.order)
    def __len__(self):
        return len(self.order)

def _find_subsequence(seq, pattern):
    if not pattern or len(pattern) > len(seq):
        return None
    last = len(seq) - len(pattern) + 1
    for start in range(last):
        if seq[start:start + len(pattern)] == pattern:
            return start
    return None

assistant_prefix_ids = tokenizer(
    ASSISTANT_PREFIX, add_special_tokens=False,
).input_ids
print(f"Prompt loss weight: {PROMPT_LOSS_WEIGHT}")
print(f"Assistant prefix token ids: {assistant_prefix_ids}")

# --- Sanity assertion: ensure the assistant-prefix id sequence we look for
#     during loss masking actually appears, contiguously, inside a fully
#     rendered chat template. If isolated-prefix tokenization disagrees with
#     in-context tokenization (special-token boundary handling, leading-space
#     merges, etc.), `_find_subsequence` would silently return None for every
#     row at training time and the soft prompt-loss mask would degrade into a
#     no-op. Fail loudly here so we never ship a full-loss run by accident.
_demo_text = tokenizer.apply_chat_template(
    records[0]["messages"], tokenize=False,
    add_generation_prompt=False, enable_thinking=False,
)
_demo_ids = tokenizer(_demo_text, add_special_tokens=False).input_ids
_hit = _find_subsequence(_demo_ids, assistant_prefix_ids)
assert _hit is not None, (
    "ASSISTANT_PREFIX tokenization does not match the rendered chat template; "
    "soft prompt-loss mask would be a no-op. Inspect tokenizer special-token "
    "handling around <|im_start|>."
)
print(f"Assistant prefix found at token offset {_hit} / total demo tokens {len(_demo_ids)}")

class StratifiedSFTTrainer(SFTTrainer):
    def __init__(self, *args, stratified_order=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.stratified_order = stratified_order
        # Running counters so we can monitor whether the soft prompt-loss mask
        # is actually applied during training. A silent rise in misses would
        # mean PROMPT_LOSS_WEIGHT effectively stops working partway through.
        self._mask_hits   = 0
        self._mask_misses = 0
        self._mask_log_every = max(
            1, self.args.logging_steps * self.args.per_device_train_batch_size,
        )
        self._mask_next_log = self._mask_log_every

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        if labels is None or PROMPT_LOSS_WEIGHT >= 1.0:
            try:
                return super().compute_loss(
                    model, inputs, return_outputs=return_outputs,
                    num_items_in_batch=num_items_in_batch,
                )
            except TypeError:
                return super().compute_loss(model, inputs, return_outputs=return_outputs)

        model_inputs = {k: v for k, v in inputs.items() if k != "labels"}
        outputs = model(**model_inputs)
        logits = outputs.logits
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        per_token_loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index=-100,
            reduction="none",
        ).view_as(shift_labels)

        weights = torch.ones_like(per_token_loss)
        for row_idx, seq in enumerate(inputs["input_ids"].detach().cpu().tolist()):
            assistant_start = _find_subsequence(seq, assistant_prefix_ids)
            if assistant_start is None:
                self._mask_misses += 1
                continue
            self._mask_hits += 1
            response_start = assistant_start + len(assistant_prefix_ids)
            prompt_shift_len = max(0, min(response_start - 1, weights.size(1)))
            if prompt_shift_len:
                weights[row_idx, :prompt_shift_len] = PROMPT_LOSS_WEIGHT

        total = self._mask_hits + self._mask_misses
        if total >= self._mask_next_log:
            print(f"[prompt-mask] hits={self._mask_hits} misses={self._mask_misses} "
                  f"({100.0 * self._mask_hits / total:.1f}% rows masked)")
            self._mask_next_log = total + self._mask_log_every

        valid = shift_labels.ne(-100)
        weights = weights * valid
        loss = (per_token_loss * weights).sum() / weights.sum().clamp_min(1.0)
        return (loss, outputs) if return_outputs else loss

    def get_train_dataloader(self):
        if self.train_dataset is None:
            raise ValueError("Trainer requires a train_dataset.")
        if self.stratified_order is None:
            return super().get_train_dataloader()
        if len(self.stratified_order) != len(self.train_dataset):
            raise ValueError("Stratified order length does not match train dataset")
        dataloader_kwargs = {
            "batch_size": self.args.per_device_train_batch_size,
            "sampler": PrecomputedOrderSampler(self.stratified_order),
            "collate_fn": self.data_collator,
            "num_workers": self.args.dataloader_num_workers,
            "pin_memory": self.args.dataloader_pin_memory,
            "persistent_workers": self.args.dataloader_persistent_workers,
            "drop_last": self.args.dataloader_drop_last,
        }
        if self.args.dataloader_num_workers > 0:
            dataloader_kwargs["prefetch_factor"] = self.args.dataloader_prefetch_factor
        return DataLoader(self.train_dataset, **dataloader_kwargs)

effective_batch_size = max(
    1, training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps,
)
stratified_order = build_stratified_index_order(record_types, effective_batch_size, SEED)
print(f"Approx stratified effective batch size: {effective_batch_size}")
print("Stratified batching by type:",
      dict(sorted(pd.Series(record_types).value_counts().to_dict().items())))

trainer = StratifiedSFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    formatting_func=formatting_prompts_func,
    stratified_order=stratified_order,
)

print("Starting SFT training...")
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    print(f"GPU memory allocated before training: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
t0 = time.time()
trainer.train()
print(f"Training done in {(time.time() - t0) / 60:.1f} min")
if torch.cuda.is_available():
    torch.cuda.synchronize()
    print(f"Peak GPU memory allocated: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print(f"[prompt-mask] final: hits={trainer._mask_hits} misses={trainer._mask_misses}")

ADAPTER_DIR = "/kaggle/working/sft_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

# ============================================================================
# §6. Build /kaggle/working/submission.zip from the freshly trained adapter
# ============================================================================
required_files = ["adapter_config.json", "adapter_model.safetensors"]
OUTPUT_DIR = "/kaggle/working"
SUBMISSION_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "submission_adapter")
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)
print(f"Packaging adapter from: {ADAPTER_DIR}")

for fname in required_files:
    src = os.path.join(ADAPTER_DIR, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing required adapter file: {src}")
    shutil.copy2(src, dst)
    print(f"Copied {fname} ({os.path.getsize(dst) / 1024 / 1024:.1f} MB)")

config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")
with open(config_path, "r") as f:
    cfg = json.load(f)
cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"]   = 0.0
with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        zf.write(os.path.join(SUBMISSION_ADAPTER_DIR, fname), fname)
        print(f"  Added {fname}")

print(f"\nsubmission.zip: {os.path.getsize(zip_path) / 1024 / 1024:.1f} MB")
print("Done! Ready to submit.")
